In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "5"

import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt
import plotnine as gg
import pandas as pd

from essential.data import load_regulondb_full
from cellbox import CellBoxEstimator

from config import get_config
from run_prediction import (
    compute_amask,
    normalize_counts,
    _as_dense,
    collect_predictions,
    _result_entries,
    compute_all_scalar_metrics,
)
from tqdm import tqdm

In [ ]:
config = get_config()

config.model.filter_regulators = True
config.model.standardize_inputs = True

In [ ]:
adata = sc.read_h5ad(config.adata_path)
adata.obs["library_size"] = adata.layers["reads"].sum(1).A1
adata = adata[adata.obs["library_size"] > config.min_library_size].copy()
if config.experiment_subset != "all":
    adata = adata[adata.obs["experiment"] == config.experiment_subset].copy()
adata = adata[adata.obs[config.perturbation_col].notna()].copy()

normalize_counts(adata, config.normalization)

ref_db = load_regulondb_full()
ref_db = ref_db.loc[lambda x: x["ri_type"].str.startswith("TF")]
_, gene_is_regulator, gene_is_target = compute_amask(adata, ref_db)
valid_features = gene_is_regulator | gene_is_target
adata = adata[:, valid_features].copy()

if config.model.filter_regulators:
    Amask, _, _ = compute_amask(adata, ref_db)
else:
    Amask = None

train_targets = list(ref_db["regulator_gene"].str.lower().unique()) + [
    config.control_key
]
pert = adata.obs[config.perturbation_col].str.lower()
is_train = pert.isin(train_targets)

adata_train = adata[is_train].copy()
adata_test = adata[~is_train].copy()
test_targets = list(
    np.intersect1d(adata_test.obs[config.perturbation_col], adata.var_names)
)
print(
    f"{adata_train.n_obs} train cells, {adata_test.n_obs} held-out cells, "
    f"{len(test_targets)} held-out targets"
)

estimator = CellBoxEstimator(
    adata_train,
    Amask,
    perturbation_col=config.perturbation_col,
    control_key=config.control_key,
    standardize_inputs=config.model.standardize_inputs,
)
estimator.fit(**config.training.to_dict())

In [ ]:
estimator.epoch_history_df

In [ ]:
estimator.epoch_history_df["val_lfc_mse"].dropna()

In [ ]:
estimator.epoch_history_df["val_loss"].plot()

In [ ]:
# adata_control = adata[pert == config.control_key].copy()
# X_gt0 = _as_dense(adata_control.X)

# X_gt, X_pred = collect_predictions(
#     test_targets, adata, adata_control, estimator, config.perturbation_col
# )
# results = _result_entries(
#     test_targets, X_gt, X_pred, X_gt0, adata, estimator, split="test"
# )

In [ ]:
results = []
adata_control = adata[pert == config.control_key].copy()
X_gt0 = _as_dense(adata_control.X)
val_targets = estimator.get_val_perturbations()
X_gt_val, X_pred_val = collect_predictions(
    val_targets, adata, adata_control, estimator, config.perturbation_col
)
results += _result_entries(
    val_targets, X_gt_val, X_pred_val, X_gt0, adata, estimator, split="val"
)

In [ ]:
ref_db = load_regulondb_full()
Apred = estimator.model.apply(
    {"params": estimator.state.params}, method=estimator.model.get_Amat
)
Apred_df = pd.DataFrame(Apred, index=adata.var_names, columns=adata.var_names)

_A_mask, gene_is_regulator, gene_is_target = compute_amask(adata, ref_db)
A_mask_df = pd.DataFrame(
    _A_mask.astype(int), index=adata.var_names, columns=adata.var_names
)

In [ ]:
# saturations = estimator.get_saturation_args(adata_train)

In [ ]:
adata_train.obs["target"].value_counts()

In [ ]:
adata_control = adata[pert == config.control_key].copy()
X0 = _as_dense(adata_control.X)
X1 = estimator.predict(adata_control, perturbation="flhC", n_steps=100)

adata_gt = adata[pert == "flhc"].copy()
delta_pred = X1 - X0
delta_gt = _as_dense(adata_gt.X).mean(0) - X0.mean(0)

In [ ]:
from scipy import stats

plt.scatter(delta_pred.mean(0), delta_gt)
print("Pearson r:", stats.pearsonr(delta_pred.mean(0), delta_gt))

In [ ]:
plt.scatter(delta_pred.mean(0), delta_gt)

In [ ]:
lfcs = pd.DataFrame(dict(lfc=delta.mean(0), gene=adata.var_names))
lfcs.sort_values("lfc")

# single example

In [ ]:
val_targets

In [ ]:
val_targets[0]

val_idx = 2

In [ ]:
mu_gt = X_gt_val[val_idx].mean(0)
mu_naive = X_gt0.mean(0)
mu_pred = X_pred_val[val_idx].mean(0)

In [ ]:
np.where(adata.var_names == "ompR")[0][0]

In [ ]:
plt.scatter(mu_pred, mu_naive, alpha=0.5)
plt.show()

plt.scatter(mu_pred, mu_gt, alpha=0.5)
plt.scatter(mu_naive, mu_gt, alpha=0.5)
plt.show()

In [ ]:
import plotly.express as px
import pandas as pd

df = pd.DataFrame(
    {
        "mu_gt": np.concatenate([mu_gt, mu_gt]),
        "predicted": np.concatenate([mu_pred, mu_naive]),
        "gene": list(adata.var_names) * 2,
        "model": ["pred"] * len(mu_gt) + ["naive"] * len(mu_gt),
    }
)
px.scatter(
    df, x="predicted", y="mu_gt", color="model", hover_name="gene", opacity=0.5
).show()

In [ ]:
A_mask_df.loc["ompC", "ompR"]

In [ ]:
gene_name = "ompF"
gene_idx = np.where(adata.var_names == gene_name)[0][0]

mu_gt[gene_idx], mu_pred[gene_idx], mu_naive[gene_idx]

In [ ]:
Apred_df.loc["dtpA", "ompR"], Apred_df.loc["ompC", "ompR"], Apred_df.loc[
    "ompF", "ompR"
],

# Check optimizaiton

In [ ]:
# adata_train_dev = adata_train[np.array(estimator.train_indices)].copy()
# adata_train_heldout = adata_train[np.array(estimator.val_indices)].copy()

adata_train_dev = estimator.adata[np.array(estimator.train_indices)].copy()
# adata_train_dev = adata_train.copy()
# adata_train_dev.X = adata_train_dev.layers["reads"].copy()
# sc.pp.normalize_total(adata_train_dev, target_sum=1e4)
# sc.pp.log1p(adata_train_dev)
target = adata_train_dev[:, "ompF"].X.toarray().flatten()
where_regulators = (A_mask_df.loc["ompF"] != 0).values
# where_regulators = np.where(adata.var_names == "ompR")[0]
regulator_names = adata.var_names[where_regulators]
X = adata_train_dev[:, where_regulators].X.toarray()

# adata_train_heldout = adata_train[np.array(estimator.val_indices)].copy()
# target_heldout = adata_train_heldout[:, "ompF"].X.toarray().flatten()
# X_heldout = adata_train_heldout[:, (A_mask_df.loc["ompF"] != 0).values].X.toarray()

In [ ]:
import sys

sys.path.append("old")
from diagnostics import fit_sigmoid, sigmoid_predict

In [ ]:
(A_mask_df.loc["ompF"] != 0).values

In [ ]:
A_mask_df.loc[["ompF"]].loc[:, regulator_names]

In [ ]:
Apred_df.loc[["ompF"]].loc[:, regulator_names]

In [ ]:
results = fit_sigmoid(X, target)
ypred = sigmoid_predict(X, results)
results

In [ ]:
coeffs = pd.DataFrame(dict(coeff=results["b"], regulator=regulator_names))
apred_coeffs = Apred_df.loc[["ompF"]].loc[:, regulator_names]

plt.scatter(coeffs["coeff"], apred_coeffs.values.flatten())
plt.xlabel("LBFGS Coefficients")
plt.ylabel("Cellbox-style coefficients")

In [ ]:
plt.scatter(X[:, 0], target, alpha=0.5)
plt.xlabel("ompR expression")
plt.ylabel("ompF expression")

In [ ]:
plt.scatter(target, ypred, alpha=0.5)

In [ ]:
target_pred = sigmoid_predict(X_heldout, results)

In [ ]:
plt.scatter(target_heldout, target_pred, alpha=0.5)

# metrics

In [ ]:
metrics = compute_all_scalar_metrics(results)

In [ ]:
test_targets = results[0]["test_targets"]
X_pred = results[0]["X_pred"]
X_gt = results[0]["X_gt"]

In [ ]:
kd_info = []
for pert_idx, target in tqdm(enumerate(test_targets)):
    gene_idx = adata.var_names.get_loc(target)

    # metrics around KD strenghth
    gene_simulated_kd = X_pred[pert_idx][:, gene_idx]
    gene_simulated_kd_strenght = (
        gene_simulated_kd.mean() / adata_control.X[:, gene_idx].mean()
    )
    print(gene_simulated_kd.mean(), adata_control.X[:, gene_idx].mean())
    gene_true_kd_strenght = (
        X_gt[pert_idx][:, gene_idx].mean() / adata_control.X[:, gene_idx].mean()
    )
    gene_sim_gt_kd_ratio = gene_simulated_kd_strenght / gene_true_kd_strenght

    # gene expression
    avg_exp_norm = adata_control.X[:, gene_idx].mean()
    avg_exp_cpm = adata_control.layers["lognormalized"][:, gene_idx].mean()
    kd_info.append(
        {
            "target": target,
            "gene_simulated_kd_strenght": gene_simulated_kd_strenght,
            "gene_true_kd_strenght": gene_true_kd_strenght,
            "gene_sim_gt_kd_ratio": gene_sim_gt_kd_ratio,
            "avg_exp_norm": avg_exp_norm,
            "avg_exp_cpm": avg_exp_cpm,
        }
    )
kd_info = pd.DataFrame(kd_info).set_index("target")

In [ ]:
kd_info["gene_simulated_kd_strenght"].describe()

In [ ]:
metrics.groupby(["split", "model_name"]).mean(numeric_only=True)

In [ ]:
pivot_table = (
    metrics.pivot_table(
        index=["target", "split"], values="lfc_mse", columns=["model_name"]
    )
    .reset_index()
    .assign(mse_ratio=lambda df: df["cellbox"] / df["mean_baseline"])
    .sort_values("mse_ratio", ascending=True)
)

In [ ]:
(gg.ggplot(pivot_table, gg.aes(x="split", y="mse_ratio")) + gg.geom_boxplot())

In [ ]:
pivot_table.sort_values("mse_ratio")

In [ ]:
", ".join(pivot_table.sort_values("mse_ratio").head(10)["target"])

In [ ]:
", ".join(pivot_table.sort_values("mse_ratio").tail(10)["target"])

In [ ]:
pivot_gene_table = (
    gene_metrics.pivot_table(
        index=["gene", "split", "metric_name", "perturbation"],
        values="metric_value",
        columns=["model_name"],
    )
    .reset_index()
    .assign(metric_ratio=lambda df: df["cellbox"] / df["mean_baseline"])
)
pivot_gene_table

In [ ]:
metric_ratio = (
    pivot_gene_table.loc[lambda x: x["split"] == "val"]
    .loc[lambda x: x["metric_name"] == "ms"]
    .groupby(["gene"])["metric_ratio"]
    .median()
    .loc[lambda x: x <= 1e6]
    .to_frame()
    .reset_index()
)
metric_ratio

In [ ]:
metric_ratio["metric_ratio"].describe()

## OLD

In [ ]:
kd_info = []
for pert_idx, target in tqdm(enumerate(test_targets)):
    gene_idx = adata.var_names.get_loc(target)

    # metrics around KD strenghth
    gene_simulated_kd = X_pred[pert_idx][:, gene_idx]
    gene_simulated_kd_strenght = (
        gene_simulated_kd.mean() / adata_control.X[:, gene_idx].mean()
    )
    gene_true_kd_strenght = (
        X_gt[pert_idx][:, gene_idx].mean() / adata_control.X[:, gene_idx].mean()
    )
    gene_sim_gt_kd_ratio = gene_simulated_kd_strenght / gene_true_kd_strenght

    # gene expression
    avg_exp_norm = adata_control.X[:, gene_idx].mean()
    avg_exp_cpm = adata_control.layers["lognormalized"][:, gene_idx].mean()
    kd_info.append(
        {
            "target": target,
            "gene_simulated_kd_strenght": gene_simulated_kd_strenght,
            "gene_true_kd_strenght": gene_true_kd_strenght,
            "gene_sim_gt_kd_ratio": gene_sim_gt_kd_ratio,
            "avg_exp_norm": avg_exp_norm,
            "avg_exp_cpm": avg_exp_cpm,
        }
    )
kd_info = pd.DataFrame(kd_info).set_index("target")

In [ ]:
shared = dict(X_gt0=X_gt0, X_gt=X_gt, test_targets=test_targets, adata=adata)
results = [
    {**shared, "X_pred": X_pred, "estimator": estimator, "model_name": "cellbox"},
    {
        **shared,
        "X_pred": X_pred_baseline,
        "estimator": None,
        "model_name": "mean_baseline",
    },
]

In [ ]:
metrics = compute_all_metrics(results)

In [ ]:
pivot_table = (
    metrics.pivot_table(index="target", values="mse_overall", columns=["model_name"])
    .reset_index()
    .assign(mse_ratio=lambda df: df["cellbox"] / df["mean_baseline"])
    .sort_values("mse_ratio", ascending=True)
    .merge(kd_info, left_on="target", right_index=True)
)

In [ ]:
(
    gg.ggplot(pivot_table, gg.aes(x="mean_baseline", y="cellbox"))
    + gg.geom_point()
    + gg.geom_abline(linetype="dashed", color="red")
)

In [ ]:
pivot_table

In [ ]:
(
    gg.ggplot(pivot_table, gg.aes(x="gene_simulated_kd_strenght", y="mse_ratio"))
    + gg.geom_point()
)

In [ ]:
(gg.ggplot(pivot_table, gg.aes(x="avg_exp_cpm", y="mse_ratio")) + gg.geom_point())

In [ ]:
genes = ", ".join(pivot_table.tail(50)["target"])
print(genes)

In [ ]:
(gg.ggplot(pivot_table, gg.aes(x="cellbox", y="mse_ratio")) + gg.geom_point())

In [ ]:
pivot_table["mse_ratio"].describe()

In [ ]:
estimator

In [ ]:
(
    gg.ggplot(pivot_table, gg.aas(x="mse_ratio"))
    + gg.geom_histogram
)